# Biodiversity Footprint Gini Calculation

Computes the population-weighted Lorenz-curve Gini coefficient of the 2023
consumption-based biodiversity footprint at four levels:

1. **Global** — one Gini over all (country, income-bin) units
2. **Country** (164) — within-country Gini across the 201 income bins
3. **Region** (7 World Bank regions) — within-region Gini across all
   (country, income-bin) units of that region's member countries
4. **Sector** — Gini of each sector class's footprint distribution across
   all (country, income-bin) units, at both the **5-class** (macro
   department) and **10-class** (detailed) classification from
   `sector_class.xlsx`



## Part 1 — Load data and define the Gini function

In [9]:
import os

import numpy as np
import pandas as pd

# ── Paths ────────────────────────────────────────────────────────────────
ACC_OUT = r'..\1_Biodiversity_footprint_accounting\output'   # category-1 output
HOU_DIR = r'E:\phdstudy\BiodiversityPHD-2ndpaper\household_expenditure'
REF_DIR = r'E:\phdstudy\reference'
OUT_DIR = r'.\output'
os.makedirs(OUT_DIR, exist_ok=True)

G, N = 164, 201   # countries, income bins

# ── Raw footprint matrices (from category 1) ────────────────────────────
fp_income = np.load(os.path.join(ACC_OUT, 'footprint_income_2023.npy'))     # (164, 201)
fp_sector = np.load(os.path.join(ACC_OUT, 'footprint_by_sector_2023.npy'))  # (120, 164, 201)
with open(os.path.join(ACC_OUT, 'country_order_2023.txt'), encoding='utf-8') as fh:
    regnam = fh.read().splitlines()

# ── Population by country x income bin (same method as category 1) ─────
mapping = pd.read_csv(os.path.join(HOU_DIR, 'GLORIA_Country_Mapping.csv'))
pop_raw = pd.read_csv(os.path.join(HOU_DIR, 'Population_by_IncomeGroup.csv'), index_col=0)
pop = np.zeros((G, N))
for gi in range(G):
    for _, row in mapping[mapping['GLORIA_Index'] == gi + 1].iterrows():
        iso3 = row['Population_ISO3']
        if pd.notna(iso3) and iso3 != 'Not in Population' and iso3 in pop_raw.columns:
            pop[gi] += pop_raw[iso3].values

# -- World Bank region, per country ---------------------------------------
# Use the same GLORIA-order WBR mapping as the N-series notebooks.
nat = (pd.read_csv(os.path.join(REF_DIR, 'nationlist_categorized_titlecase.csv'))
       .sort_values('Country_ID').reset_index(drop=True))
WBR_NAMES = {
    'EAP': 'East Asia & Pacific',
    'ECA': 'Europe & Central Asia',
    'LAC': 'Latin America & Caribbean',
    'MENA': 'Middle East & North Africa',
    'NAM': 'North America',
    'SAR': 'South Asia',
    'SSA': 'Sub-Saharan Africa',
}
REGION_ORDER = ['Sub-Saharan Africa', 'East Asia & Pacific',
                'Middle East & North Africa', 'North America',
                'Latin America & Caribbean', 'Europe & Central Asia',
                'South Asia']
region_codes = nat['WBR'].tolist()
assert len(region_codes) == G, f'WBR mapping length mismatch: {len(region_codes)} != {G}'
region_of_country = [WBR_NAMES.get(code, code) for code in region_codes]

# ── Sector classification (5-class and 10-class) ────────────────────────
sec_cls = pd.read_excel('sector_class.xlsx').sort_values('Lfd_Nr')
assert len(sec_cls) == fp_sector.shape[0], 'sector_class.xlsx must list all 120 sectors'
class5_of_sector = sec_cls['5class'].values     # (120,)
class10_of_sector = sec_cls['class'].values      # (120,)
CLASS5_ORDER = sorted(sec_cls['5class'].unique())
CLASS10_ORDER = sorted(sec_cls['class'].unique())


def gini_lorenz(footprint, population):
    """Population-weighted Lorenz-curve Gini coefficient.

    Units (flattened footprint/population arrays, one entry per (country,
    bin) cell) are ranked by per-capita footprint ascending; Gini = 1 - 2 x
    (area under the Lorenz curve).
    """
    f, p = footprint.ravel().astype(float), population.ravel().astype(float)
    mask = (p > 0) & np.isfinite(f) & (f >= 0)
    f, p = f[mask], p[mask]
    if len(f) < 2 or p.sum() == 0:
        return np.nan
    if f.sum() == 0:
        return 0.0
    percap = np.divide(f, p, out=np.zeros_like(f), where=p > 0)
    order = np.argsort(percap, kind='mergesort')
    f, p = f[order], p[order]
    lx = np.concatenate([[0], np.cumsum(p) / p.sum()])
    ly = np.concatenate([[0], np.cumsum(f) / f.sum()])
    return float(1 - 2 * np.trapz(ly, lx))


print(f'Loaded fp_income {fp_income.shape}, fp_sector {fp_sector.shape}, pop {pop.shape}')
print(f'5-class sectors: {CLASS5_ORDER}')
print(f'10-class sectors: {CLASS10_ORDER}')


Loaded fp_income (164, 201), fp_sector (120, 164, 201), pop (164, 201)
5-class sectors: ['Crops', 'Forestry and logging', 'Industrial products', 'Livestock and fishery', 'Services']
10-class sectors: ['Fishery', 'Forestry and logging', 'Fruits and vegetables', 'Grains', 'Livestock', 'Manufacturing products', 'Mining products', 'Other crops', 'Services', 'Utilities and construction works']


## Part 2 — Compute Gini at each level and export

In [10]:
# ── 1. Global Gini ───────────────────────────────────────────────────────
gini_global = gini_lorenz(fp_income, pop)

# ── 2. Country Gini (within-country, across income bins) ───────────────
df_country = pd.DataFrame({
    'Country': regnam,
    'Gini': [gini_lorenz(fp_income[i], pop[i]) for i in range(G)],
    'Population': pop.sum(axis=1),
})

# ── 3. Region Gini (within-region, across member countries x bins) ─────
df_region = pd.DataFrame({
    'Region': REGION_ORDER,
    'Gini': [gini_lorenz(fp_income[[j for j, r in enumerate(region_of_country) if r == reg]],
                          pop[[j for j, r in enumerate(region_of_country) if r == reg]])
             for reg in REGION_ORDER],
})

# ── 4. Sector Gini, 5-class and 10-class ────────────────────────────────
# Each class's footprint is summed over its member sectors, distributed across
# the same (country, bin) population units as the global/region Gini above.
def sector_class_gini(class_of_sector, order):
    rows = []
    for cls in order:
        sec_idx = np.where(class_of_sector == cls)[0]
        fp_cls = fp_sector[sec_idx].sum(axis=0)      # (164, 201)
        rows.append({'Class': cls, 'Gini': gini_lorenz(fp_cls, pop),
                     'Footprint_PDF': fp_cls.sum()})
    return pd.DataFrame(rows).sort_values('Gini', ascending=False).reset_index(drop=True)


df_sector5 = sector_class_gini(class5_of_sector, CLASS5_ORDER)
df_sector10 = sector_class_gini(class10_of_sector, CLASS10_ORDER)

# ── Export ───────────────────────────────────────────────────────────────
out_xlsx = os.path.join(OUT_DIR, 'gini_summary_2023.xlsx')
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    pd.DataFrame({'Metric': ['Global_Gini'], 'Value': [gini_global]}).to_excel(w, sheet_name='Global', index=False)
    df_country.to_excel(w, sheet_name='Country', index=False)
    df_region.to_excel(w, sheet_name='Region', index=False)
    df_sector5.to_excel(w, sheet_name='Sector_5class', index=False)
    df_sector10.to_excel(w, sheet_name='Sector_10class', index=False)

print(f'Global Gini: {gini_global:.4f}')
print(f'\nRegion Gini:\n{df_region.to_string(index=False)}')
print(f'\nSector 5-class Gini:\n{df_sector5.to_string(index=False)}')
print(f'\nSector 10-class Gini:\n{df_sector10.to_string(index=False)}')
print(f'\nSaved: {out_xlsx}')


Global Gini: 0.5717

Region Gini:
                    Region     Gini
        Sub-Saharan Africa 0.740923
       East Asia & Pacific 0.577951
Middle East & North Africa 0.551599
             North America 0.431527
 Latin America & Caribbean 0.397385
     Europe & Central Asia 0.377056
                South Asia 0.302111

Sector 5-class Gini:
                Class     Gini  Footprint_PDF
 Forestry and logging 0.803663       0.006917
             Services 0.748804       0.018729
  Industrial products 0.681569       0.009736
Livestock and fishery 0.656838       0.004867
                Crops 0.527658       0.025430

Sector 10-class Gini:
                           Class     Gini  Footprint_PDF
                 Mining products 0.864100       0.000612
            Forestry and logging 0.803663       0.006917
Utilities and construction works 0.767559       0.001355
                        Services 0.748804       0.018729
                         Fishery 0.693705       0.000205
          Manuf